In [ ]:
library(ggplot2)
library(behavr)
library(scopr)
library(sleepr)
library(ggetho)
library(plotly)
#library(survival)
library(cowplot)
#library(ggthemes)
library(plotly)
library(data.table)
library(stringi)
library(ggtern)
library(ggpubr)
library(EnvStats)
library(RColorBrewer)
library(dplyr)
library(plyr)

In [ ]:
REMOTE_DATA_SOURCE <- "ftp://turing.lab.gilest.ro/auto_generated_data/ethoscope_results/"
MY_DIR <- "/home/hjones/insecticide_movement/"
setwd(MY_DIR)

# This is the placement of the data in this computer, make sure this is a file where you want it saved!
DATA_DIR <- "/mnt/ethoscope_results"
#This is the placement of the real data, on the NAS

#this is the place were a cache version of the data is stored, once it has been taken from NAS. This make the loading faster on the second time.
CACHE <- "/home/cache"

#This is the query of the experiment (this is the table you made of the data)
METADATA <- "sleep_data/metadata_ethoscope_r.csv"

In [ ]:
#To get the files from the remote source
query <- link_ethoscope_metadata(METADATA,
                                 result_dir = DATA_DIR)

In [ ]:
#This is the magic step, it loads the data to R and applies a function at the same time, in this case, the asleep annotation.
dt <- load_ethoscope(query,
                     reference_hour = 9.0, 
                     FUN = sleep_annotation,
                     cache = CACHE)

In [ ]:
#to include baseline days if there are multiple conditions 
dt[,t:=t+days(xmv(baseline_days))]

In [ ]:
#to check whether there are dead animals.
dt_curated <- curate_dead_animals(dt)
summary(dt_curated)

In [ ]:
#looking only at 3h of the day (baseline - no SD or rebound)
dt_SD_first_3 <- dt_curated[t >days(8) & t< days(8.125)]

In [ ]:
#subsetting for sdi - subset and then run each subset with below code
dt_control <-dt_curated[xmv(sdi)=="0" ]
dt_1 <-dt_curated[xmv(sdi)=="1" ]
dt_2 <-dt_curated[xmv(sdi)=="2" ]
dt_3 <-dt_curated[xmv(sdi)=="3" ]
dt_4 <-dt_curated[xmv(sdi)=="4" ]
dt_5 <-dt_curated[xmv(sdi)=="5" ]
dt_6 <-dt_curated[xmv(sdi)=="6" ]
dt_7 <-dt_curated[xmv(sdi)=="7" ]
dt_8 <-dt_curated[xmv(sdi)=="8" ]
dt_9 <-dt_curated[xmv(sdi)=="9" ]
dt_10 <-dt_curated[xmv(sdi)=="10" ]

In [ ]:
dt_control <- dt_control[dt_control, meta=T]

In [ ]:
#make table with only max velocity data by id
dt_control_velocity <- dt_control[, .(max_velocity), by=id]

In [ ]:
#quick summary to check the data
summary(dt_control_velocity)

In [ ]:
#puts the data into a list
split.df <- split(dt_day_velocity, dt_day_velocity$id)

In [ ]:
for(i in 1:length(split.df)){
  write.csv(split.df[[i]], paste0("insecticides/sleep_data/control/sdi_control_",i,
                                  ".csv"))
}

In [ ]:
#to remove all empty files in a folder
## Get vector of all file names
ff <- dir("insecticides/sleep_data/control/", recursive=TRUE, full.names=TRUE)
## Extract vector of empty files' names
eff <- ff[file.info(ff)[["size"]]==23]
## Remove empty files
unlink(eff, recursive=TRUE, force=FALSE)